In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np

from tqdm import tqdm
from torchvision import transforms
import os
import pandas as pd

from HARUnet_model_v2_1 import HARU_net

from ResUNet_model import ResUNet

from utils import psnr, batch_psnr

import torch.ao.quantization as tq
torch.backends.quantized.engine = "fbgemm"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
class EarlyStopping:
    def __init__(self, patience=10, delta=0):
        """
        Args:
            patience (int): How many epochs to wait after last improvement.
            delta (float): Minimum change to consider an improvement.
        """
        self.patience = patience
        self.delta = delta
        self.best_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            print(f'Best loss = {self.best_loss}')
            self.counter = 0

In [3]:
# Load data
def load_data(pickle_file):
    with open(pickle_file, 'rb') as f:
        patches = pickle.load(f)
    return patches  # Expecting a NumPy array (N, 1, H, W)

class CBCTDataset(Dataset):
    def __init__(self, noisy_patches, target_patches):
        self.noisy = noisy_patches  # Keep as is
        self.target = target_patches  # Keep as is

    def __len__(self):
        return len(self.noisy)

    def __getitem__(self, idx):
        noisy_tensor = torch.tensor(self.noisy[idx], dtype=torch.float32)
        target_tensor = torch.tensor(self.target[idx], dtype=torch.float32)

        return noisy_tensor, target_tensor
    

In [4]:
pickled_train_inputs = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Train_noisyCBCT_patches.pkl"
pickled_train_targets = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Train_origCBCT_patches.pkl"
train_inputs = load_data(pickled_train_inputs)  # Shape: (N, 1, H, W)
train_targets = load_data(pickled_train_targets)  # Shape: (N, 1, H, W)
batch_size = 16
train_dataset = CBCTDataset(train_inputs, train_targets)
train_loader = DataLoader(train_dataset, batch_size, shuffle=True, num_workers=0, pin_memory=True)

pickled_val_inputs = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Val_noisyCBCT_patches.pkl"
pickled_val_targets = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Val_origCBCT_patches.pkl"
val_inputs = load_data(pickled_val_inputs)  # Shape: (N, 1, H, W)
val_targets = load_data(pickled_val_targets)  # Shape: (N, 1, H, W)

val_dataset = CBCTDataset(val_inputs, val_targets)
val_loader = DataLoader(val_dataset, batch_size, shuffle=True, num_workers=0, pin_memory=True)

In [5]:
class DistillationLoss(nn.Module):
    def __init__(self):
        super().__init__()
        #self.logit_alpha = nn.Parameter(torch.tensor(0.0))
        #self.lamda = lamda
        self.hard = nn.MSELoss()
        self.soft = nn.MSELoss()

    def forward(self, student_out, teacher, targets, alpha):
        
        #alpha = torch.sigmoid(self.logit_alpha)
        hard_loss = self.hard(student_out, targets)
        soft_loss = self.soft(student_out, teacher)

        return alpha * soft_loss + (1 - alpha) * hard_loss
    
criterion_distillation = DistillationLoss()
criterion_inference = nn.MSELoss()

In [6]:
model_dir = r"C:\Users\au711969\OneDrive - Aarhus universitet\Dentistry_Stuff\My Research projects\CBCT Denoising Project\BM3D_on_CBCT\Codes"

teacher_modelname = r"HARUnetv2_11_trainedon_noisytorawCBCTs_CadavarData_at_42epochs_.pth"
teacher_model = torch.load(os.path.join(model_dir,teacher_modelname)).to(device)
teacher_model = teacher_model.to(device)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False

C:\Users\au711969\AppData\Local\Temp\ipykernel_23688\3572041785.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher_model = torch.load(os.path.join(model_dir,teacher

In [7]:
student_model = HARU_net(dim=64, hab_heads=8, hab_ws=16, hab_mlp_ratio=2.0, hab_depth=6)
student_modelname_ = "HARUnetv2_11_trainedon_noisytorawCBCTs_CadavarData_at_42epochs_.pth"
student_model_ = torch.load(os.path.join(model_dir,student_modelname_),map_location="cpu")

if isinstance(student_model_ , torch.nn.DataParallel):
    student_model_  = student_model_.module

student_model.load_state_dict(student_model_.state_dict())

import torch.ao.quantization as tq

student_model.train()
qconfig = tq.get_default_qat_qconfig("fbgemm")
student_model.qconfig = qconfig
print(student_model.qconfig)  # sanity check

student_model_qat = tq.prepare_qat(student_model, inplace=False)
student_model_qat.to(device)


C:\Users\au711969\AppData\Local\Temp\ipykernel_23688\2102781083.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  student_model_ = torch.load(os.path.join(model_dir,studen

QConfig(activation=functools.partial(<class 'torch.ao.quantization.fake_quantize.FusedMovingAvgObsFakeQuantize'>, observer=<class 'torch.ao.quantization.observer.MovingAverageMinMaxObserver'>, quant_min=0, quant_max=255, reduce_range=True){}, weight=functools.partial(<class 'torch.ao.quantization.fake_quantize.FusedMovingAvgObsFakeQuantize'>, observer=<class 'torch.ao.quantization.observer.MovingAveragePerChannelMinMaxObserver'>, quant_min=-128, quant_max=127, dtype=torch.qint8, qscheme=torch.per_channel_symmetric){})


c:\Users\au711969\AppData\Local\anaconda3\envs\KnTorch\Lib\site-packages\torch\ao\quantization\observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


HARU_net(
  (ConvBlock1): ConvBlock(
    (block): Sequential(
      (0): Conv2d(
        1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
        (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
          fake_quant_enabled=tensor([1], device='cuda:0'), observer_enabled=tensor([1], device='cuda:0'), scale=tensor([1.], device='cuda:0'), zero_point=tensor([0], device='cuda:0', dtype=torch.int32), dtype=torch.qint8, quant_min=-128, quant_max=127, qscheme=torch.per_channel_symmetric, reduce_range=False
          (activation_post_process): MovingAveragePerChannelMinMaxObserver(min_val=tensor([], device='cuda:0'), max_val=tensor([], device='cuda:0'))
        )
        (activation_post_process): FusedMovingAvgObsFakeQuantize(
          fake_quant_enabled=tensor([1], device='cuda:0'), observer_enabled=tensor([1], device='cuda:0'), scale=tensor([1.], device='cuda:0'), zero_point=tensor([0], device='cuda:0', dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=127, qscheme=t

In [8]:
optimizer = optim.Adam(student_model.parameters(), lr=1e-6)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=5,
    threshold=1e-8,      # consider changes smaller than this as "no improvement"
    threshold_mode='rel',
    cooldown=0,
    min_lr=1e-10,
    verbose=True
)

c:\Users\au711969\AppData\Local\anaconda3\envs\KnTorch\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [9]:
early_stopping = EarlyStopping(patience=10, delta=1e-7)
    
train_loss_history = []
train_inpsnr_history = []
train_outpsnr_history = []

#train_issim_history = []
#train_ossim_history = []

val_loss_history = []
val_inpsnr_history = []
val_outpsnr_history = []

#val_issim_history = []
#val_ossim_history = []


epochs = 20
for epoch in range(epochs):
    student_model_qat.train()

    train_loss = 0.0
    train_inpsnr = 0
    train_outpsnr = 0
    
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    alpha = 0.5

    for train_inputs, train_targets in train_loop:
        
        train_inputs, train_targets = train_inputs.unsqueeze(1).to(device), train_targets.unsqueeze(1).to(device)

        with torch.no_grad():
            teacher_outputs = teacher_model(train_inputs)

        optimizer.zero_grad()      
        train_outputs = student_model_qat(train_inputs)
        
        batch_loss = criterion_distillation(train_outputs, teacher_outputs, train_targets, alpha)
        
        batch_loss.backward()
        
        optimizer.step()  

        train_loss += batch_loss.item()
        train_inpsnr += batch_psnr(train_inputs, train_targets)
        train_outpsnr += batch_psnr(train_outputs, train_targets)

    train_loss /= len(train_loader)
    epoch_train_inpsnr = train_inpsnr / len(train_loader)
    epoch_train_outpsnr = train_outpsnr / len(train_loader)

    train_loss_history.append(train_loss)
    train_inpsnr_history.append(epoch_train_inpsnr)
    train_outpsnr_history.append(epoch_train_outpsnr)

    student_model_qat.eval()
    val_loss = 0.0
    val_inpsnr = 0
    val_outpsnr = 0
    count = 0
    
    for val_inputs, val_targets in val_loader:

        val_inputs, val_targets = val_inputs.unsqueeze(1).to(device), val_targets.unsqueeze(1).to(device)

        with torch.no_grad():
            val_outputs = student_model_qat(val_inputs)
        if batch_psnr(val_inputs, val_targets) != float('inf'):
            val_loss += criterion_inference(val_outputs, val_targets).item()
            val_inpsnr += batch_psnr(val_inputs, val_targets)
            val_outpsnr += batch_psnr(val_outputs, val_targets)
        else:
            count = count + 1

    val_loss /= (len(val_loader)-count)
    epoch_val_inpsnr = val_inpsnr / (len(val_loader)-count)
    epoch_val_outpsnr = val_outpsnr / (len(val_loader)-count)

    val_loss_history.append(val_loss)
    val_inpsnr_history.append(epoch_val_inpsnr) 
    val_outpsnr_history.append(epoch_val_outpsnr)

    train_loop.set_postfix(train_loss=train_loss)
    #if early_stopping.best_loss is None or val_loss < early_stopping.best_loss:
    torch.save(student_model_qat.state_dict(), "QAT_DistilledHARUnet__best_1.pth")

    print(f'Epoch [{epoch+1}/{epochs}], Train loss: {train_loss:.8f}, Input PSNR: {epoch_train_inpsnr:.3f}, Output PSNR: {epoch_train_outpsnr:.3f}, Alpha: {alpha:.3f}') 
    print(f'............, Validation loss: {val_loss:.8f}, Validation Input PSNR: {epoch_val_inpsnr:.3f}, Validation Output PSNR: {epoch_val_outpsnr:.3f}')

    if early_stopping.early_stop:
        print("Early stopping triggered.")
        total_training_epochs = epoch+1
        break

Epoch 1/20:   0%|          | 0/3127 [00:12<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 8.00 GiB. GPU 0 has a total capacity of 47.99 GiB of which 22.69 GiB is free. Of the allocated memory 23.32 GiB is allocated by PyTorch, and 20.81 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Assuming 'model' is your trained model
torch.save(student_model_qat.state_dict(), f'QAT_DistilledHARUnet_trainedon_CBCT_CadavarData_at_{epoch+1}epochs_new.pth')
#model = torch.load("BM3D-LUnet_trainedon_CBCT_CadavarData_at_91epochs_lr001.pth")

In [ ]:
pickled_test_inputs = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Test_noisyCBCT_patches.pkl"
pickled_test_targets = r"D:\CBCT_Denoising\Pickled CBCT Data\MoreNoisy\Test_origCBCT_patches.pkl"
test_inputs = load_data(pickled_test_inputs)  # Shape: (N, 1, H, W)
test_targets = load_data(pickled_test_targets)  # Shape: (N, 1, H, W)

test_dataset = CBCTDataset(test_inputs, test_targets)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=0,pin_memory=True)

In [ ]:
student_model.eval()
test_loss = 0.0
test_inpsnr = 0
test_outpsnr = 0
test_BM3Drltvpsnr = 0
for test_inputs, test_targets in test_loader:

    test_inputs, test_targets = test_inputs.unsqueeze(1).to(device), test_targets.unsqueeze(1).to(device)
    optimizer.zero_grad()
            
    with torch.no_grad():
        test_outputs = student_model_qat(test_inputs)
    if batch_psnr(test_inputs, test_targets) != float('inf'):
        test_loss += criterion_inference(test_outputs, test_targets).item()
        test_inpsnr += batch_psnr(test_inputs, test_targets)
        print(f'Batch_psnr = {batch_psnr(test_inputs, test_targets)}, total test_inpsnr = {test_inpsnr}')
        test_outpsnr += batch_psnr(test_outputs, test_targets)
        

test_loss /= (len(test_loader)-1)
epoch_test_inpsnr = test_inpsnr /(len(test_loader)-1)
epoch_test_outpsnr = test_outpsnr / (len(test_loader)-1)
epoch_test_BM3Drltvpsnr = test_BM3Drltvpsnr / (len(test_loader)-1)
        

print(f'............, Test loss: {test_loss:.8f}, Test Target PSNR: {epoch_test_inpsnr:.3f}, Test outPSNR: {epoch_test_outpsnr:.3f}')
#print(f'Validation loss: {val_loss:.8f}, Validation inPSNR: {epoch_val_inpsnr:.3f}, Validation outPSNR: {epoch_val_outpsnr:.3f}')

Batch_psnr = 19.545089119803315, total test_inpsnr = 19.545089119803315
Batch_psnr = 20.06487379082635, total test_inpsnr = 39.60996291062966
Batch_psnr = 19.275247425351374, total test_inpsnr = 58.885210335981036
Batch_psnr = 19.609761512177077, total test_inpsnr = 78.49497184815812
Batch_psnr = 20.65669817613614, total test_inpsnr = 99.15167002429425
Batch_psnr = 19.624506792849495, total test_inpsnr = 118.77617681714375
Batch_psnr = 21.35767696806927, total test_inpsnr = 140.13385378521303
Batch_psnr = 19.04809098967199, total test_inpsnr = 159.18194477488504
Batch_psnr = 20.116742506431834, total test_inpsnr = 179.29868728131686
Batch_psnr = 19.87495181054399, total test_inpsnr = 199.17363909186085
Batch_psnr = 19.613679255577118, total test_inpsnr = 218.78731834743797
Batch_psnr = 19.15049643709813, total test_inpsnr = 237.9378147845361
Batch_psnr = 19.965234819264722, total test_inpsnr = 257.9030496038008
Batch_psnr = 19.515093282165395, total test_inpsnr = 277.4181428859662
Batc

In [ ]:
# Assuming loss_history and psnr_history are your lists of metrics
data = {
    'Epoch': range(1, len(train_loss_history) + 1),  # Start epoch count from 1
    'Training Loss': train_loss_history,
    'Training BM3D PSNR': train_inpsnr_history,
    'Training outPSNR': train_outpsnr_history,
    'Validation Loss': val_loss_history,
    'Validation BM3D PSNR': val_inpsnr_history,
    'Validation outPSNR': val_outpsnr_history,
    'Testing Loss': test_loss,
    'Testing BM3D PSNR': epoch_test_inpsnr,
    'Testing outPSNR': epoch_test_outpsnr,
}

# Create a DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv('QAT_TrainingnTesting_metrics_propDestilledHARUnet_epochs{epoch}_.csv', index=False)  # Set index=False to avoid saving row indices  

: 

: 

: 

: 